# LeetCode 2323 — Find Minimum Time to Finish All Jobs II bằng Python

Link đề: https://leetcode.com/problems/find-minimum-time-to-finish-all-jobs-ii/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

> **Bài Premium:** đề bài trong notebook được viết theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc và chữ ký hàm.

Skeleton LeetCode:

```python
class Solution:
    def minimumTime(self, jobs: List[int], workers: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa: ghép **một-một** `n` công việc với `n` công nhân; thời gian của một cặp là `⌈job / worker⌉` ngày; tối thiểu hóa **thời gian lớn nhất** (bài toán ghép cặp nút cổ chai — bottleneck assignment).
2. Chứng minh tham lam **sắp xếp cả hai rồi ghép theo chỉ số** bằng lập luận hoán đổi (exchange argument).
3. Một góc nhìn khác: **tìm nhị phân theo số ngày `D`** + kiểm tra khả thi (công nhân `w` làm được việc `j` trong `D` ngày ⇔ `j ≤ D·w`).
4. Tính `⌈a / b⌉` bằng số nguyên.
5. Thấy giới hạn của tham lam: nó tối ưu cho **max**, nhưng không chắc tối ưu cho **tổng** `Σ ⌈a/b⌉` (mục 15.2, kiểm chứng bằng vét cạn).

## 2. Đề bài & ví dụ

Cho hai mảng cùng độ dài `jobs` và `workers`: `jobs[i]` là lượng thời gian cần để hoàn thành việc `i`, `workers[j]` là lượng thời gian công nhân `j` làm được **mỗi ngày**. Mỗi việc giao cho đúng một công nhân, mỗi công nhân làm đúng một việc. Trả về **số ngày nhỏ nhất** để mọi việc hoàn thành.

**Ví dụ 1:** `jobs = [5,2,4]`, `workers = [1,7,5]` → `2`.

**Ví dụ 2:** `jobs = [3,18,15,9]`, `workers = [6,5,1,3]` → `3`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `1 ≤ jobs[i], workers[i] ≤ 10⁵`.

In [1]:
from typing import List
from itertools import permutations
from bisect import bisect_left
import random, time

examples = [([5, 2, 4], [1, 7, 5], 2), ([3, 18, 15, 9], [6, 5, 1, 3], 3)]
for j, w, exp in examples:
    print(f"jobs={j} workers={w} -> kỳ vọng {exp}")

jobs=[5, 2, 4] workers=[1, 7, 5] -> kỳ vọng 2
jobs=[3, 18, 15, 9] workers=[6, 5, 1, 3] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1 Thời gian của một cặp: `⌈job / worker⌉`

Công nhân làm `w` đơn vị mỗi ngày ⇒ sau `d` ngày làm được `d·w`. Hoàn thành việc `j` ⇔ `d·w ≥ j` ⇔ `d ≥ j / w` ⇔ `d ≥ ⌈j / w⌉`. Trần nguyên: `(j + w − 1) // w`.

In [2]:
for j, w in [(5, 7), (2, 1), (4, 5), (18, 6), (15, 5), (9, 3), (3, 1)]:
    print(f"job {j:>2}, worker {w}: {(j + w - 1) // w} ngày")

job  5, worker 7: 1 ngày
job  2, worker 1: 2 ngày
job  4, worker 5: 1 ngày
job 18, worker 6: 3 ngày
job 15, worker 5: 3 ngày
job  9, worker 3: 3 ngày
job  3, worker 1: 3 ngày


### 3.1b Hai cách viết trần nguyên và vì sao đúng

`(a + b − 1) // b`: viết `a = qb + r` (`0 ≤ r < b`); nếu `r = 0` được `q`, nếu `r ≥ 1` thì `qb + r + b − 1 ∈ [(q+1)b, (q+2)b)` ⇒ `q + 1`. `-(-a // b)`: phép `//` của Python làm tròn **xuống** (về −∞), nên `−a // b = −⌈a/b⌉`. Hai cách cho cùng kết quả với mọi `a ≥ 0`, `b ≥ 1`.

In [3]:
assert all((a + b - 1) // b == -(-a // b) for a in range(0, 500) for b in range(1, 60))
print("hai cách viết trần trùng nhau; ví dụ -(-7 // 2) =", -(-7 // 2), ", -7 // 2 =", -7 // 2)

hai cách viết trần trùng nhau; ví dụ -(-7 // 2) = 4 , -7 // 2 = -4


### 3.2 Hàm `⌈a / b⌉` đơn điệu

Tăng theo `a`, giảm theo `b`. Đây là tính chất duy nhất cần cho chứng minh tham lam.

### 3.3 Mục tiêu "max nhỏ nhất" (bottleneck)

Khác với "tổng nhỏ nhất" (bài toán ghép cặp Hungarian), mục tiêu max chỉ quan tâm cặp **tệ nhất**. Bài toán bottleneck tổng quát giải bằng tìm nhị phân + ghép cặp hai phía; ở đây cấu trúc đơn điệu cho phép tham lam đơn giản.

## 4. Ý tưởng nền tảng / trực giác

Việc **nặng nhất** nên giao cho công nhân **mạnh nhất**, việc nặng thứ nhì cho công nhân mạnh thứ nhì, … Nếu giao việc nặng cho người yếu và việc nhẹ cho người mạnh, đổi chéo hai người không thể làm tệ hơn.

Tính tay ví dụ 2: sắp `jobs = [3, 9, 15, 18]`, `workers = [1, 3, 5, 6]`.

| cặp | job | worker | ⌈job/worker⌉ |
|---|---|---|---|
| 1 | 3 | 1 | 3 |
| 2 | 9 | 3 | 3 |
| 3 | 15 | 5 | 3 |
| 4 | 18 | 6 | 3 |

`max = 3` ✔. Ví dụ 1: `jobs = [2, 4, 5]`, `workers = [1, 5, 7]` ⇒ `⌈2/1⌉ = 2`, `⌈4/5⌉ = 1`, `⌈5/7⌉ = 1` ⇒ `2` ✔.

In [4]:
for j, w, exp in examples:
    J, W = sorted(j), sorted(w)
    days = [(a + b - 1) // b for a, b in zip(J, W)]
    print(f"jobs {J} | workers {W} | ngày {days} -> max {max(days)} (kỳ vọng {exp})")

jobs [2, 4, 5] | workers [1, 5, 7] | ngày [2, 1, 1] -> max 2 (kỳ vọng 2)
jobs [3, 9, 15, 18] | workers [1, 3, 5, 6] | ngày [3, 3, 3, 3] -> max 3 (kỳ vọng 3)


### 4.1 Vì sao không ghép "việc khó nhất với người mạnh nhất, còn lại tùy ý"?

Ghép đúng cặp lớn nhất là chưa đủ — **mọi** cặp đều có thể là nút cổ chai. Ví dụ `jobs = [6, 5, 1]`, `workers = [1, 3, 6]`: nếu chỉ chắc chắn `6 ↔ 6` rồi ghép `5 ↔ 1`, `1 ↔ 3` thì cặp `5 ↔ 1` mất 5 ngày; ghép đủ thứ tự `1 ↔ 1`, `5 ↔ 3`, `6 ↔ 6` chỉ mất 2 ngày.

In [5]:
j, w = [6, 5, 1], [1, 3, 6]
bad_pairs = [(6, 6), (5, 1), (1, 3)]
good_pairs = list(zip(sorted(j), sorted(w)))
for name, ps in [("chỉ ghép cặp lớn nhất", bad_pairs), ("ghép đủ thứ tự", good_pairs)]:
    print(f"{name:<22}: {[(a, b, (a + b - 1) // b) for a, b in ps]} -> max {max((a + b - 1) // b for a, b in ps)}")

chỉ ghép cặp lớn nhất : [(6, 6, 1), (5, 1, 5), (1, 3, 1)] -> max 5
ghép đủ thứ tự        : [(1, 1, 1), (5, 3, 2), (6, 6, 1)] -> max 2


## 5. Cách 1 — Brute Force / baseline

### 5.1 Thử mọi hoán vị ghép cặp — O(n! · n)

In [6]:
def brute(jobs, workers):
    return min(max((a + b - 1) // b for a, b in zip(jobs, perm)) for perm in permutations(workers))

for j, w, exp in examples:
    print(brute(j, w), "OK" if brute(j, w) == exp else "SAI")

2 OK
3 OK


`n = 10⁵` ⇒ `10⁵!` — chỉ dùng kiểm chứng với `n ≤ 8`.

### 5.2 Baseline 2 — Tìm nhị phân số ngày + ghép cặp hai phía (bipartite matching) tổng quát

Với `D` cố định, cạnh `(việc i, công nhân k)` tồn tại nếu `⌈jobs[i] / workers[k]⌉ ≤ D`. Khả thi ⇔ có ghép cặp hoàn hảo. Dùng thuật toán Kuhn (O(V·E)) — tổng quát cho mọi hàm chi phí, nhưng chậm (O(n³ log)).

In [7]:
def binary_search_matching(jobs, workers):
    n = len(jobs)
    def feasible(D):
        adj = [[k for k in range(n) if (jobs[i] + workers[k] - 1) // workers[k] <= D] for i in range(n)]
        match = [-1] * n
        def try_job(i, seen):
            for k in adj[i]:
                if not seen[k]:
                    seen[k] = True
                    if match[k] == -1 or try_job(match[k], seen):
                        match[k] = i; return True
            return False
        return all(try_job(i, [False] * n) for i in range(n))
    lo, hi = 1, max(jobs)
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid): hi = mid
        else: lo = mid + 1
    return lo

for j, w, exp in examples:
    print(binary_search_matching(j, w), "OK" if binary_search_matching(j, w) == exp else "SAI")

2 OK
3 OK


**Nút thắt:** đồ thị khả thi có cấu trúc **lồng nhau** — công nhân mạnh hơn làm được mọi việc mà công nhân yếu hơn làm được (tập việc của họ lồng nhau). Với cấu trúc này ghép cặp hoàn hảo tồn tại ⇔ ghép theo thứ tự sắp xếp thành công ⇒ không cần thuật toán ghép cặp tổng quát.

## 6. Ý tưởng thuật toán / bước đột phá

```
sort(jobs); sort(workers)
return max(ceil(jobs[i] / workers[i]) for i)
```

**Lập luận hoán đổi.** Xét hai việc `a ≤ a'` và hai công nhân `b ≤ b'`. Ghép "chéo" `(a, b')`, `(a', b)` cho nút cổ chai `max(⌈a/b'⌉, ⌈a'/b⌉) ≥ ⌈a'/b⌉`. Ghép "thẳng" `(a, b)`, `(a', b')`: `⌈a/b⌉ ≤ ⌈a'/b⌉` (vì `a ≤ a'`) và `⌈a'/b'⌉ ≤ ⌈a'/b⌉` (vì `b ≤ b'`) ⇒ `max(…) ≤ ⌈a'/b⌉`. Vậy ghép thẳng **không tệ hơn**. Lặp lại việc gỡ mọi cặp chéo ⇒ ghép theo thứ tự sắp xếp là tối ưu.

**Độ phức tạp:** O(n log n) do sắp xếp; bộ nhớ O(n) (bản sao đã sắp).

### 6.1 Cấu trúc lồng nhau: kiểm chứng "ghép hoàn hảo tồn tại ⇔ ghép theo thứ tự thành công"

Với `D` cố định, công nhân `W` làm được tập việc `{ j : j ≤ D·W }` — các tập này **lồng nhau** theo `W`. Ta so sánh điều kiện đơn giản `J_sorted[i] ≤ D·W_sorted[i]` với thuật toán ghép cặp tổng quát (Kuhn) trên nhiều `D` và dữ liệu ngẫu nhiên:

In [8]:
def perfect_matching_exists(jobs, workers, D):
    n = len(jobs); adj = [[k for k in range(n) if jobs[i] <= D * workers[k]] for i in range(n)]
    match = [-1] * n
    def aug(i, seen):
        for k in adj[i]:
            if not seen[k]:
                seen[k] = True
                if match[k] == -1 or aug(match[k], seen): match[k] = i; return True
        return False
    return all(aug(i, [False] * n) for i in range(n))

rng = random.Random(61); checked = 0
for _ in range(400):
    n = rng.randint(1, 7); j = [rng.randint(1, 40) for _ in range(n)]; w = [rng.randint(1, 8) for _ in range(n)]
    for D in range(1, 12):
        greedy_ok = all(a <= D * b for a, b in zip(sorted(j), sorted(w)))
        assert greedy_ok == perfect_matching_exists(j, w, D); checked += 1
print(checked, "cặp (dữ liệu, D): điều kiện tham lam == tồn tại ghép hoàn hảo")

4400 cặp (dữ liệu, D): điều kiện tham lam == tồn tại ghép hoàn hảo


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp và ghép theo chỉ số (cách doocs)

In [9]:
def sort_match(jobs, workers):
    J, W = sorted(jobs), sorted(workers)
    return max((a + b - 1) // b for a, b in zip(J, W))

for j, w, exp in examples:
    print(sort_match(j, w), "OK" if sort_match(j, w) == exp else "SAI")

2 OK
3 OK


### 7.2 Cách 3 — Tìm nhị phân `D` + kiểm tra tham lam (minh họa cấu trúc lồng nhau)

Với `D` cố định, công nhân `w` làm được việc có độ lớn `≤ D·w`. Sắp xếp cả hai; khả thi ⇔ `jobs_sorted[i] ≤ D · workers_sorted[i]` với mọi `i`.

In [10]:
def binary_search_greedy(jobs, workers):
    J, W = sorted(jobs), sorted(workers)
    def feasible(D):
        return all(a <= D * b for a, b in zip(J, W))
    lo, hi = 1, max(J)
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid): hi = mid
        else: lo = mid + 1
    return lo

for j, w, exp in examples:
    print(binary_search_greedy(j, w), "OK" if binary_search_greedy(j, w) == exp else "SAI")

2 OK
3 OK


`hi = max(jobs)` là đủ vì mọi công nhân có `w ≥ 1` ⇒ trong `max(jobs)` ngày làm xong mọi việc.

### 7.3 Cách 4 — Đếm phân phối thay cho sắp xếp (giá trị ≤ 10⁵) — O(n + V)

In [11]:
def counting_sort_match(jobs, workers):
    V = max(max(jobs), max(workers))
    def csort(a):
        cnt = [0] * (V + 1)
        for x in a: cnt[x] += 1
        out = []
        for v in range(V + 1):
            if cnt[v]: out.extend([v] * cnt[v])
        return out
    J, W = csort(jobs), csort(workers)
    return max((a + b - 1) // b for a, b in zip(J, W))

for j, w, exp in examples:
    print(counting_sort_match(j, w), "OK" if counting_sort_match(j, w) == exp else "SAI")

2 OK
3 OK


### 7.4 Cách 5 — Bản trả về cả **phép phân công** (chỉ số gốc)

In [12]:
def assignment(jobs, workers):
    jo = sorted(range(len(jobs)), key=jobs.__getitem__)
    wo = sorted(range(len(workers)), key=workers.__getitem__)
    pairs = list(zip(jo, wo))                       # (chỉ số việc, chỉ số công nhân)
    days = max((jobs[i] + workers[k] - 1) // workers[k] for i, k in pairs)
    return days, sorted(pairs)

for j, w, exp in examples:
    d, pairs = assignment(j, w)
    print(d, [f"việc {i} ({j[i]}) -> công nhân {k} ({w[k]}): {(j[i] + w[k] - 1) // w[k]} ngày" for i, k in pairs])

2 ['việc 0 (5) -> công nhân 1 (7): 1 ngày', 'việc 1 (2) -> công nhân 0 (1): 2 ngày', 'việc 2 (4) -> công nhân 2 (5): 1 ngày']
3 ['việc 0 (3) -> công nhân 2 (1): 3 ngày', 'việc 1 (18) -> công nhân 0 (6): 3 ngày', 'việc 2 (15) -> công nhân 1 (5): 3 ngày', 'việc 3 (9) -> công nhân 3 (3): 3 ngày']


## 8. TRACE từng bước

### 8.1 Trace lập luận hoán đổi: bắt đầu từ một phép ghép tùy ý, gỡ dần các cặp chéo

In [13]:
def bottleneck(jobs, workers_order):
    return max((a + b - 1) // b for a, b in zip(jobs, workers_order))

rng = random.Random(8)
J = sorted([5, 2, 4, 9, 7]); Wp = [1, 7, 5, 3, 2]; rng.shuffle(Wp)
print("jobs (đã sắp):", J, "| ghép ban đầu:", Wp, "-> max", bottleneck(J, Wp))
changed = True
while changed:
    changed = False
    for i in range(len(J) - 1):
        if Wp[i] > Wp[i + 1]:                         # cặp chéo: việc nhẹ hơn có công nhân mạnh hơn
            before = bottleneck(J, Wp)
            Wp[i], Wp[i + 1] = Wp[i + 1], Wp[i]
            after = bottleneck(J, Wp)
            assert after <= before
            print(f"  đổi vị trí {i},{i+1}: max {before} -> {after}, ghép {Wp}")
            changed = True
print("cuối:", Wp, "max", bottleneck(J, Wp), "| brute:", brute(J, [1, 7, 5, 3, 2]))

jobs (đã sắp): [2, 4, 5, 7, 9] | ghép ban đầu: [3, 1, 2, 5, 7] -> max 4
  đổi vị trí 0,1: max 4 -> 3, ghép [1, 3, 2, 5, 7]
  đổi vị trí 1,2: max 3 -> 2, ghép [1, 2, 3, 5, 7]
cuối: [1, 2, 3, 5, 7] max 2 | brute: 2


Mỗi lần đổi một cặp chéo kề nhau, nút cổ chai **không tăng** (đúng như chứng minh), và cuối cùng ta được ghép theo thứ tự sắp xếp.

### 8.2 Trace tìm nhị phân `D` trên ví dụ 2

In [14]:
J, W = sorted(examples[1][0]), sorted(examples[1][1])
lo, hi = 1, max(J)
while lo < hi:
    mid = (lo + hi) // 2
    ok = all(a <= mid * b for a, b in zip(J, W))
    print(f"[{lo}, {hi}] thử D = {mid}: {'khả thi' if ok else 'không'}  (sức chứa {[mid * b for b in W]} vs việc {J})")
    if ok: hi = mid
    else: lo = mid + 1
print("D =", lo)

[1, 18] thử D = 9: khả thi  (sức chứa [9, 27, 45, 54] vs việc [3, 9, 15, 18])
[1, 9] thử D = 5: khả thi  (sức chứa [5, 15, 25, 30] vs việc [3, 9, 15, 18])
[1, 5] thử D = 3: khả thi  (sức chứa [3, 9, 15, 18] vs việc [3, 9, 15, 18])
[1, 3] thử D = 2: không  (sức chứa [2, 6, 10, 12] vs việc [3, 9, 15, 18])
D = 3


### 8.3 Đối chiếu phép phân công với lời giải trong đề (ví dụ 1)

Đề phân công: công nhân 2 (5/ngày) → việc 0 (5), công nhân 0 (1/ngày) → việc 1 (2), công nhân 1 (7/ngày) → việc 2 (4). Cách sắp xếp cho:

In [15]:
j, w, _ = examples[0]
d, pairs = assignment(j, w)
for i, k in pairs:
    print(f"việc {i} (cần {j[i]}) <- công nhân {k} ({w[k]}/ngày): {(j[i] + w[k] - 1) // w[k]} ngày")
print("max =", d, "| phép phân công của đề cho:", max((5 + 4) // 5, (2 + 0) // 1, (4 + 6) // 7))

việc 0 (cần 5) <- công nhân 1 (7/ngày): 1 ngày
việc 1 (cần 2) <- công nhân 0 (1/ngày): 2 ngày
việc 2 (cần 4) <- công nhân 2 (5/ngày): 1 ngày
max = 2 | phép phân công của đề cho: 2


Sắp xếp ghép việc 2 (4) với công nhân 2 (5) và việc 0 (5) với công nhân 1 (7) — khác lời giải của đề nhưng cùng nút cổ chai 2 (do cặp `việc 1 ↔ công nhân 0` bắt buộc: công nhân yếu nhất phải nhận một việc nào đó, và việc nhẹ nhất cho 2 ngày). Nhiều phép phân công cùng tối ưu.

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def minimumTime(self, jobs: List[int], workers: List[int]) -> int:
        jobs = sorted(jobs)
        workers = sorted(workers)
        # việc nặng thứ i ghép với công nhân mạnh thứ i
        return max((a + b - 1) // b for a, b in zip(jobs, workers))

In [17]:
sol = Solution()
for j, w, exp in examples:
    got = sol.minimumTime(j, w)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

2 | kỳ vọng 2 | OK
3 | kỳ vọng 3 | OK


## 10. Vì sao thuật toán đúng

Ký hiệu `t(a, b) = ⌈a / b⌉`, không giảm theo `a`, không tăng theo `b`.

**Bổ đề (hoán đổi).** Với `a ≤ a'`, `b ≤ b'`: `max(t(a, b), t(a', b')) ≤ max(t(a, b'), t(a', b))`.
*Chứng minh:* `t(a, b) ≤ t(a', b)` và `t(a', b') ≤ t(a', b)` ⇒ vế trái `≤ t(a', b) ≤` vế phải.

**Định lý.** Gọi `J₁ ≤ … ≤ Jₙ`, `W₁ ≤ … ≤ Wₙ`. Phép ghép `Jᵢ ↔ Wᵢ` đạt nút cổ chai nhỏ nhất.
*Chứng minh:* lấy phép ghép tối ưu `σ` (việc `i` ↔ công nhân `W_{σ(i)}`). Nếu `σ` không phải đồng nhất, tồn tại `i < i'` với `σ(i) > σ(i')` (một **nghịch thế**). Áp bổ đề với `a = Jᵢ ≤ a' = J_{i'}`, `b = W_{σ(i')} ≤ b' = W_{σ(i)}`: đổi chỗ hai công nhân không làm tăng max của hai cặp đó, các cặp khác không đổi ⇒ phép ghép mới vẫn tối ưu và có ít nghịch thế hơn. Sau hữu hạn bước được phép ghép đồng nhất, vẫn tối ưu. ∎

**Tìm nhị phân.** `feasible(D)` đơn điệu theo `D`; với `D` cố định, công nhân `Wᵢ` làm được mọi việc `≤ D·Wᵢ` — các tập này lồng nhau theo `i`; theo định lý Hall cho cấu trúc lồng nhau (hoặc trực tiếp theo bổ đề hoán đổi), ghép hoàn hảo tồn tại ⇔ `Jᵢ ≤ D·Wᵢ` với mọi `i`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hoán vị | O(n! · n) | O(n) |
| 1b. Tìm nhị phân + Kuhn | O(log V · n · n²) | O(n²) |
| 2. Sắp xếp + ghép | O(n log n) | O(n) |
| 3. Tìm nhị phân + tham lam | O(n log n + n log V) | O(n) |
| 4. Đếm phân phối | O(n + V) | O(V) |
| 5. Trả về phân công | O(n log n) | O(n) |

In [18]:
rng = random.Random(2323)
J = [rng.randint(1, 10**5) for _ in range(10**5)]; W = [rng.randint(1, 10**5) for _ in range(10**5)]
res = []
for name, f in [("sort_match", sort_match), ("binary_search_greedy", binary_search_greedy),
                ("counting_sort_match", counting_sort_match), ("Solution", Solution().minimumTime)]:
    t0 = time.perf_counter(); res.append(f(J, W)); dt = time.perf_counter() - t0
    print(f"{name:<21} {dt*1000:7.1f} ms -> {res[-1]}")
assert len(set(res)) == 1

sort_match               48.9 ms -> 2


binary_search_greedy    195.1 ms -> 2


counting_sort_match      32.7 ms -> 2
Solution                 48.0 ms -> 2


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng — đối chiếu lại đề gốc nếu có Premium)

In [19]:
ALL = {"brute": brute, "binary_search_matching": binary_search_matching, "sort_match": sort_match,
       "binary_search_greedy": binary_search_greedy, "counting_sort_match": counting_sort_match,
       "assignment": lambda j, w: assignment(j, w)[0], "Solution": Solution().minimumTime}
edge = examples + [
    ([1], [1], 1),
    ([10**5], [1], 10**5),                     # việc lớn nhất, người yếu nhất
    ([1], [10**5], 1),                         # luôn ít nhất 1 ngày
    ([6, 6], [3, 2], 3),                       # 6/3 = 2, 6/2 = 3
    ([7, 1], [7, 1], 1),                       # ghép đúng thứ tự: mỗi cặp 1 ngày
    ([10, 9, 8], [3, 3, 3], 4),
]
for j, w, exp in edge:
    for name, f in ALL.items():
        got = f(list(j), list(w))
        assert got == exp, (name, j, w, got, exp)
    print(f"jobs={j!s:<12} workers={w!s:<12} -> {exp}")
print("edge OK")

jobs=[5, 2, 4]    workers=[1, 7, 5]    -> 2
jobs=[3, 18, 15, 9] workers=[6, 5, 1, 3] -> 3
jobs=[1]          workers=[1]          -> 1
jobs=[100000]     workers=[1]          -> 100000
jobs=[1]          workers=[100000]     -> 1
jobs=[6, 6]       workers=[3, 2]       -> 3
jobs=[7, 1]       workers=[7, 1]       -> 1
jobs=[10, 9, 8]   workers=[3, 3, 3]    -> 4
edge OK


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [20]:
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(1, 6)
    j = [rng.randint(1, rng.choice([5, 30, 10**5])) for _ in range(n)]; w = [rng.randint(1, rng.choice([5, 30, 10**5])) for _ in range(n)]
    ref = brute(j, w)
    for name, f in ALL.items():
        assert f(list(j), list(w)) == ref, (name, j, w)
print("400 bộ ngẫu nhiên OK")

400 bộ ngẫu nhiên OK


### 12.3 Tính chất: `1 ≤ ans ≤ ⌈max(jobs)/min(workers)⌉`; `ans ≥ max_i ⌈J_i / W_max⌉`; nhân mọi công nhân lên 2 không làm tăng đáp án

In [21]:
rng = random.Random(7)
for _ in range(500):
    n = rng.randint(1, 30)
    j = [rng.randint(1, 1000) for _ in range(n)]; w = [rng.randint(1, 1000) for _ in range(n)]
    ans = Solution().minimumTime(j, w)
    assert 1 <= ans <= -(-max(j) // min(w))
    assert ans >= -(-max(j) // max(w))
    assert Solution().minimumTime(j, [2 * x for x in w]) <= ans
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị

In [22]:
n = 10**5
assert Solution().minimumTime([10**5] * n, [1] * n) == 10**5
assert Solution().minimumTime([1] * n, [10**5] * n) == 1
assert Solution().minimumTime(list(range(1, n + 1)), list(range(1, n + 1))) == 1      # ghép i <-> i
assert Solution().minimumTime(list(range(1, n + 1)), list(range(n, 0, -1))) == 1      # thứ tự gốc không quan trọng
print("cực trị OK")

cực trị OK


Trường hợp `jobs = workers = [1..n]` nhưng đưa vào theo thứ tự ngược nhau: ghép theo chỉ số gốc sẽ cho `⌈n / 1⌉ = n` ngày, còn sắp xếp cả hai cho 1 ngày — minh họa vì sao Lỗi 3 nghiêm trọng.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp **ngược chiều** (việc nặng nhất cho người yếu nhất)

In [23]:
def wrong_opposite(jobs, workers):
    J, W = sorted(jobs), sorted(workers, reverse=True)        # SAI
    return max((a + b - 1) // b for a, b in zip(J, W))

for j, w, _ in examples:
    print("bản sai :", wrong_opposite(j, w), "| bản đúng:", Solution().minimumTime(j, w))

bản sai : 5 | bản đúng: 2
bản sai : 18 | bản đúng: 3


**Sửa:** sắp **cùng chiều** cả hai mảng.

### Lỗi 2 — Dùng chia **sàn** `a // b`

In [24]:
def wrong_floor(jobs, workers):
    J, W = sorted(jobs), sorted(workers)
    return max(a // b for a, b in zip(J, W))                  # SAI: việc chưa xong vẫn cần thêm một ngày

for j, w in [(examples[0][0], examples[0][1]), (examples[1][0], examples[1][1]), ([4], [5]), ([7, 3], [2, 2])]:
    print("bản sai :", wrong_floor(j, w), "| bản đúng:", Solution().minimumTime(j, w), "cho", j, w)

bản sai : 2 | bản đúng: 2 cho [5, 2, 4] [1, 7, 5]
bản sai : 3 | bản đúng: 3 cho [3, 18, 15, 9] [6, 5, 1, 3]
bản sai : 0 | bản đúng: 1 cho [4] [5]
bản sai : 3 | bản đúng: 4 cho [7, 3] [2, 2]


**Vì sao sai:** cả hai ví dụ đề **trùng hợp** đúng: ở ví dụ 1 cặp quyết định là `⌈2/1⌉ = ⌊2/1⌋ = 2`, ở ví dụ 2 mọi cặp đều chia hết. Nhưng `[4], [5]`: `⌊4/5⌋ = 0` — công nhân 5 đơn vị/ngày không thể làm xong việc 4 đơn vị trong 0 ngày; `[7, 3], [2, 2]`: `⌊7/2⌋ = 3` trong khi cần 4 ngày. **Sửa:** trần `(a + b − 1) // b`.

### Lỗi 3 — Không sắp xếp (ghép theo chỉ số gốc)

In [25]:
def wrong_unsorted(jobs, workers):
    return max((a + b - 1) // b for a, b in zip(jobs, workers))  # SAI: phải chọn phép ghép

for j, w, _ in examples:
    print("bản sai :", wrong_unsorted(j, w), "| bản đúng:", Solution().minimumTime(j, w))

bản sai : 5 | bản đúng: 2
bản sai : 15 | bản đúng: 3


### Lỗi 4 — Ước lượng bằng tổng: `⌈Σ jobs / Σ workers⌉`

In [26]:
def wrong_total(jobs, workers):
    return -(-sum(jobs) // sum(workers))                      # SAI: mỗi người chỉ làm MỘT việc

for j, w in [(examples[0][0], examples[0][1]), (examples[1][0], examples[1][1]), ([10, 10], [1, 10])]:
    print("bản sai :", wrong_total(j, w), "| bản đúng:", Solution().minimumTime(j, w), "cho", j, w)

bản sai : 1 | bản đúng: 2 cho [5, 2, 4] [1, 7, 5]
bản sai : 3 | bản đúng: 3 cho [3, 18, 15, 9] [6, 5, 1, 3]
bản sai : 2 | bản đúng: 10 cho [10, 10] [1, 10]


**Vì sao sai:** công thức tổng giả định công việc chia nhỏ được cho nhiều người. Ví dụ 1: `11 / 13` ⇒ 1, nhưng thật là 2. Ví dụ 2 **trùng hợp** đúng (`45 / 15 = 3`). `[10, 10], [1, 10]`: tổng cho `⌈20/11⌉ = 2`, nhưng công nhân yếu (1/ngày) buộc phải làm trọn một việc 10 ⇒ 10 ngày.

### Lỗi 5 — Dùng `math.ceil(a / b)` với float (đúng ở giới hạn đề, nhưng sai khi số lớn)

In [27]:
import math
def float_ceil(a, b): return math.ceil(a / b)
bad = None
rng = random.Random(5)
for _ in range(10**5):
    b = rng.randint(1, 1000); q = rng.randint(10**15, 10**16); a = q * b
    if float_ceil(a, b) != (a + b - 1) // b: bad = (a, b); break
print("trong giới hạn đề (<= 10^5) float luôn đúng:", all(float_ceil(a, b) == (a + b - 1) // b for a in range(1, 300) for b in range(1, 300)))
a, b = bad
print("với số lớn:", a, b, "-> float", float_ceil(a, b), "| nguyên", (a + b - 1) // b)

trong giới hạn đề (<= 10^5) float luôn đúng: True
với số lớn: 2655813007602353263 287 -> float 9253703859241648 | nguyên 9253703859241649


**Sửa:** luôn dùng trần nguyên — không tốn thêm gì và đúng với mọi kích thước số.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ghép một-một hai dãy, chi phí cặp **đơn điệu** theo từng biến, mục tiêu **max** hoặc **tổng** của hàm "tốt" ⇒ thử **sắp xếp cùng chiều / ngược chiều** và chứng minh bằng hoán đổi hai cặp.
- "Tối thiểu hóa giá trị lớn nhất" ⇒ cũng có thể **tìm nhị phân đáp án** + kiểm tra khả thi.
- Trần nguyên `(a + b − 1) // b`.
- Khi hàm chi phí là `g(f(a, b))` với `g` đơn điệu (ở đây `g = ⌈·⌉`), tối ưu max của `g(f)` đạt tại cùng phép ghép tối ưu max của `f` — giải bản liên tục rồi làm tròn (mục 15.4).
- Bài cùng họ: 2071 Maximum Number of Tasks You Can Assign, 826 Most Profit Assigning Work, 455 Assign Cookies, 1723 Find Minimum Time to Finish All Jobs (mỗi người nhiều việc — NP-khó), 2285, 870 Advantage Shuffle.

## 15. Biến thể của bài

### 15.1 Biến thể A — Nhiều công nhân hơn việc (`m ≥ n`): chọn `n` công nhân mạnh nhất

In [28]:
def more_workers(jobs, workers):
    J = sorted(jobs); W = sorted(workers)[-len(jobs):]
    return max((a + b - 1) // b for a, b in zip(J, W))

def more_workers_brute(jobs, workers):
    return min(max((a + b - 1) // b for a, b in zip(jobs, perm)) for perm in permutations(workers, len(jobs)))

rng = random.Random(151)
for _ in range(300):
    n = rng.randint(1, 4); m = rng.randint(n, 6)
    j = [rng.randint(1, 30) for _ in range(n)]; w = [rng.randint(1, 30) for _ in range(m)]
    assert more_workers(j, w) == more_workers_brute(j, w)
print(more_workers([5, 2, 4], [1, 7, 5, 2, 9]))

1


### 15.2 Biến thể B — Tối thiểu hóa **tổng** số ngày `Σ ⌈a/b⌉`: ghép cùng chiều **không** luôn tối ưu

Nếu bỏ phép lấy trần, mục tiêu `Σ aᵢ · (1/bᵢ)` là một tích vô hướng; theo bất đẳng thức sắp xếp lại, tổng nhỏ nhất đạt khi ghép `a` lớn với `1/b` nhỏ, tức `a` lớn với `b` lớn — cũng là ghép cùng chiều. Nhưng phép **lấy trần** phá vỡ lập luận này (trần của từng số hạng không cộng tính); vét cạn tìm phản ví dụ:

In [29]:
def sum_sorted(j, w):
    return sum((a + b - 1) // b for a, b in zip(sorted(j), sorted(w)))

def sum_brute(j, w):
    return min(sum((a + b - 1) // b for a, b in zip(j, p)) for p in permutations(w))

rng = random.Random(152); found = None
for _ in range(20000):
    n = rng.randint(2, 4); j = [rng.randint(1, 12) for _ in range(n)]; w = [rng.randint(1, 6) for _ in range(n)]
    if sum_sorted(j, w) != sum_brute(j, w): found = (j, w); break
print("phản ví dụ cho mục tiêu tổng:", found, "| cùng chiều:", sum_sorted(*found), "| tối ưu:", sum_brute(*found))

phản ví dụ cho mục tiêu tổng: ([6, 8, 6, 5], [2, 5, 6, 4]) | cùng chiều: 9 | tối ưu: 7


Bài học: chứng minh hoán đổi của bản gốc dựa vào mục tiêu **max**; đổi mục tiêu phải chứng minh lại (hoặc dùng thuật toán ghép cặp có trọng số).

### 15.3 Biến thể C — Mỗi công nhân làm **nhiều** việc (LC 1723) — NP-khó, cần vét cạn/bitmask

In [30]:
from functools import lru_cache
def jobs_multi(jobs, k):
    n = len(jobs); ssum = [sum(jobs[i] for i in range(n) if m >> i & 1) for m in range(1 << n)]
    @lru_cache(maxsize=None)
    def f(t, mask):
        if t == 1: return ssum[mask]
        best = ssum[mask]; sub = mask
        while sub:
            best = min(best, max(ssum[sub], f(t - 1, mask ^ sub))); sub = (sub - 1) & mask
        return best
    return f(k, (1 << n) - 1)
print(jobs_multi([3, 2, 3], 3), jobs_multi([1, 2, 4, 7, 8], 2))

3 11


Khác biệt then chốt: ràng buộc "một-một" của 2323 cho phép tham lam; bỏ ràng buộc đó, bài trở thành phân hoạch tập (giống 2305).

### 15.4 Biến thể D — Thời gian là **số thực** `a / b` (được làm phần ngày): tối thiểu hóa max bằng phân số chính xác

Lập luận hoán đổi chỉ dùng tính đơn điệu, nên ghép cùng chiều vẫn tối ưu; dùng `Fraction` để so sánh chính xác.

In [31]:
from fractions import Fraction
def exact_ratio(jobs, workers):
    return max(Fraction(a, b) for a, b in zip(sorted(jobs), sorted(workers)))

def exact_ratio_brute(jobs, workers):
    return min(max(Fraction(a, b) for a, b in zip(jobs, p)) for p in permutations(workers))

rng = random.Random(154)
for _ in range(300):
    n = rng.randint(1, 5); j = [rng.randint(1, 30) for _ in range(n)]; w = [rng.randint(1, 10) for _ in range(n)]
    assert exact_ratio(j, w) == exact_ratio_brute(j, w)
    assert -(-exact_ratio(j, w).numerator // exact_ratio(j, w).denominator) == Solution().minimumTime(j, w)
print(exact_ratio(*examples[0][:2]), exact_ratio(*examples[1][:2]))

2 3


Dòng assert thứ hai cho thấy: đáp án bài gốc đúng bằng **trần** của đáp án bản số thực (vì `⌈·⌉` đơn điệu, `max ⌈xᵢ⌉ = ⌈max xᵢ⌉`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hoán vị | định nghĩa | O(n!·n) | O(n) | kiểm chứng |
| 1b. Tìm nhị phân + Kuhn | bottleneck tổng quát | O(n³ log V) | O(n²) | hàm chi phí không đơn điệu |
| 2. Sắp xếp + ghép | hoán đổi | O(n log n) | O(n) | **mặc định** |
| 3. Tìm nhị phân + tham lam | cấu trúc lồng nhau | O(n log n) | O(n) | minh họa / mở rộng |
| 4. Đếm phân phối | V nhỏ | O(n + V) | O(V) | tối ưu tiệm cận |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumTime([4, 4], [2, 4])` và `minimumTime([9, 1, 1], [1, 1, 9])`?

**Bài 2 (tính tay):** `jobs = [12, 5, 7, 20]`, `workers = [3, 4, 2, 5]`. Sắp xếp, ghép, tính từng cặp và đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(jobs, workers, D)` trả về `True` ⇔ hoàn thành được trong `D` ngày.

**Bài 4 (tự viết lại):** Viết `bt4_todo(jobs, workers)` cài bản nộp không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(jobs, workers)` trả về số **công nhân tối thiểu cần thay bằng công nhân mạnh gấp đôi** để đáp án giảm ít nhất 1 ngày (vét cạn nhỏ cũng được, `n ≤ 7`).

In [32]:
def bt3_todo(jobs, workers, D):
    # TODO: khả thi trong D ngày?
    pass

def bt4_todo(jobs, workers):
    # TODO: bản nộp
    pass

def bt5_todo(jobs, workers):
    # TODO: số công nhân cần nâng cấp
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: sắp `[4, 4]`, `[2, 4]`. Mức 2: `⌈4/2⌉ = 2`, `⌈4/4⌉ = 1`. Mức 3: `2`; câu hai: `[1,1,9]` ↔ `[1,1,9]` ⇒ `1`.

**Bài 2.** Mức 1: `[5, 7, 12, 20]`, `[2, 3, 4, 5]`. Mức 2: `3, 3, 3, 4`. Mức 3: `4`.

**Bài 3.** Mức 1: sắp xếp. Mức 2: so `J[i] ≤ D·W[i]`. Mức 3: `all(...)`.

**Bài 4.** Mức 1: sắp hai mảng. Mức 2: trần nguyên. Mức 3: `max`.

**Bài 5.** Mức 1: thử mọi tập con công nhân được nâng cấp theo kích thước tăng dần. Mức 2: với mỗi tập, nhân đôi `workers` tương ứng và tính lại. Mức 3: trả kích thước nhỏ nhất đạt mục tiêu (hoặc −1 nếu đáp án đã là 1).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2`, `1`. **Bài 2:** `4`.

In [33]:
from itertools import combinations
assert Solution().minimumTime([4, 4], [2, 4]) == 2 and Solution().minimumTime([9, 1, 1], [1, 1, 9]) == 1
assert Solution().minimumTime([12, 5, 7, 20], [3, 4, 2, 5]) == 4

def bt3(jobs, workers, D):
    return all(a <= D * b for a, b in zip(sorted(jobs), sorted(workers)))

def bt4(jobs, workers):
    return sort_match(jobs, workers)

def bt5(jobs, workers):
    base = Solution().minimumTime(jobs, workers)
    if base == 1: return -1
    n = len(workers)
    for r in range(1, n + 1):
        for S in combinations(range(n), r):
            w2 = [2 * w if i in S else w for i, w in enumerate(workers)]
            if Solution().minimumTime(jobs, w2) <= base - 1: return r
    return -1

rng = random.Random(1998)
for _ in range(300):
    n = rng.randint(1, 6); j = [rng.randint(1, 40) for _ in range(n)]; w = [rng.randint(1, 10) for _ in range(n)]
    ans = brute(j, w)
    assert bt3(j, w, ans) and (ans == 1 or not bt3(j, w, ans - 1))
    assert bt4(j, w) == ans
    r = bt5(j, w)
    assert r == -1 if ans == 1 else 1 <= r <= n          # nhân đôi mọi công nhân luôn giảm được ít nhất một ngày khi ans >= 2
print(bt5([12, 5, 7, 20], [3, 4, 2, 5]), bt5(*examples[1][:2]))
print("Đáp án bài tập: tất cả assert OK")

1 4
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
jobs, workers = sorted(jobs), sorted(workers)          # cùng chiều
return max((a + b - 1) // b for a, b in zip(jobs, workers))
```

**Checklist:**
- [ ] Sắp **cả hai** mảng **cùng chiều**.
- [ ] Trần nguyên, không chia sàn, không float.
- [ ] Mục tiêu là **max**, không phải tổng.
- [ ] Đáp án luôn ≥ 1.
- [ ] (Premium) Đối chiếu lại đề gốc.